# 🎬 Bilibili 影片下載 + Markdown 轉換器

自動下載 Bilibili 播放清單影片，提取音訊，使用 Whisper 轉錄，生成 Markdown 檔案。

**作者**：蝦仔  **日期**：2025-03-05

## Step 1: 連接 Google Drive

運行以下代碼，授權連接你嘅 Google Drive。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 設定工作目錄
WORK_DIR = "/content/drive/MyDrive/bilibili-projects"
!mkdir -p {WORK_DIR}
%cd {WORK_DIR}
print(f"✅ 已連接 Google Drive: {WORK_DIR}")

## Step 2: 安裝依賴

安裝必要嘅 Python 套件。

In [ ]:
# 安裝 yt-dlp
!pip install -q yt-dlp

# 安裝 Whisper
!pip install -q openai-whisper

# 安裝其他工具
!pip install -q ffmpeg-python

# 安裝 FFmpeg
!apt-get update -qq
!apt-get install -y -qq ffmpeg

print("✅ 依賴安裝完成！")

## Step 3: 配置播放清單

修改以下網址為你想下載嘅 Bilibili 播放清單。

In [ ]:
# 設定播放清單 URL
PLAYLIST_URL = "https://space.bilibili.com/1925268550/lists/839662?type=season"  # 修改呢度

# Whisper 模型選擇: tiny, base, small, medium, large
WHISPER_MODEL = "small"

# 語言: zh (中文), en (英文), ja (日文), auto (自動檢測)
LANGUAGE = "zh"

# 限制處理數量 (設為 None 處理全部, 或設為數字如 5 只處理前5條)
LIMIT = 5

print(f"📝 配置:")
print(f"  播放清單: {PLAYLIST_URL}")
print(f"  Whisper 模型: {WHISPER_MODEL}")
print(f"  語言: {LANGUAGE}")
print(f"  限制: {LIMIT if LIMIT else '無限制'}")

## Step 4: 下載影片

自動下載播放清單中嘅影片。

In [ ]:
import os
import json
import subprocess
from pathlib import Path

# 創建目錄
DOWNLOAD_DIR = f"{WORK_DIR}/downloads"
AUDIO_DIR = f"{WORK_DIR}/audio"
OUTPUT_DIR = f"{WORK_DIR}/output"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_playlist_info(url):
    """獲取播放清單資訊"""
    cmd = [
        "yt-dlp",
        "--flat-playlist",
        "--dump-json",
        "--no-warnings",
        url
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    videos = []
    for line in result.stdout.strip().split('\n'):
        if line:
            try:
                data = json.loads(line)
                videos.append({
                    'id': data.get('id', ''),
                    'title': data.get('title', '未知標題'),
                    'uploader': data.get('uploader', '未知UP主'),
                    'duration': data.get('duration', 0),
                    'webpage_url': data.get('webpage_url', '')
                })
            except:
                pass
    return videos

# 獲取播放清單
print("🔍 正在解析播放清單...")
videos = get_playlist_info(PLAYLIST_URL)
print(f"✅ 找到 {len(videos)} 條影片")

# 應用限制
if LIMIT:
    videos = videos[:LIMIT]
    print(f"📌 只處理前 {LIMIT} 條影片")

# 顯示列表
for i, v in enumerate(videos, 1):
    print(f"{i}. {v['title']} ({v['uploader']})")

In [ ]:
# 下載影片
def download_video(url, filename):
    output_path = f"{DOWNLOAD_DIR}/{filename}.%(ext)s"
    cmd = [
        "yt-dlp",
        "--format", "best[ext=mp4]/best",
        "--output", output_path,
        "--no-warnings",
        url
    ]
    subprocess.run(cmd, check=True)
    
    # 找到下載嘅檔案
    for f in os.listdir(DOWNLOAD_DIR):
        if f.startswith(filename):
            return os.path.join(DOWNLOAD_DIR, f)
    return None

# 逐個下載
downloaded = []
for i, video in enumerate(videos, 1):
    print(f"\n🎥 [{i}/{len(videos)}] 下載: {video['title']}")
    
    safe_name = "".join(c for c in video['title'] if c.isalnum() or c in " -_")[:50]
    filename = f"{i:03d}_{safe_name}"
    
    try:
        path = download_video(video['webpage_url'], filename)
        if path:
            downloaded.append({
                'video': video,
                'path': path,
                'filename': filename
            })
            print(f"✅ 完成: {path}")
    except Exception as e:
        print(f"❌ 失敗: {e}")

print(f"\n📊 成功下載: {len(downloaded)}/{len(videos)}")

## Step 5: 提取音軌並轉錄

使用 Whisper 將語音轉為文字。

In [ ]:
import whisper

# 載入 Whisper 模型
print(f"🤖 載入 Whisper 模型: {WHISPER_MODEL}")
model = whisper.load_model(WHISPER_MODEL)
print("✅ 模型載入完成！")

In [ ]:
def extract_audio(video_path, output_name):
    """提取音軌"""
    output_path = f"{AUDIO_DIR}/{output_name}.wav"
    cmd = [
        "ffmpeg",
        "-i", video_path,
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        "-ac", "1",
        "-y",
        output_path
    ]
    subprocess.run(cmd, capture_output=True, check=True)
    return output_path

def transcribe(audio_path):
    """轉錄音訊"""
    result = model.transcribe(audio_path, language=LANGUAGE)
    return result

def generate_markdown(video_info, transcript, index, output_path):
    """生成 Markdown"""
    content = f"""# {video_info['title']}

## 元數據

| 項目 | 內容 |
|------|------|
| **來源** | Bilibili |
| **標題** | {video_info['title']} |
| **UP主** | {video_info['uploader']} |
| **時長** | {video_info['duration']} 秒 |
| **連結** | {video_info['webpage_url']} |

---

## 內容轉錄

"""
    
    for seg in transcript['segments']:
        start = int(seg['start'] // 60), int(seg['start'] % 60)
        end = int(seg['end'] // 60), int(seg['end'] % 60)
        content += f"### {start[0]:02d}:{start[1]:02d} - {end[0]:02d}:{end[1]:02d}\n\n"
        content += f"{seg['text'].strip()}\n\n"
    
    content += "---\n\n*由 bilibili-to-markdown 自動生成*\n"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(content)
    
    return output_path

# 處理每個下載嘅影片
results = []
for item in downloaded:
    print(f"\n🎵 [{item['video']['title']}]")
    
    # 提取音軌
    print("  提取音軌...")
    audio_path = extract_audio(item['path'], item['filename'])
    
    # 轉錄
    print("  轉錄中...")
    transcript = transcribe(audio_path)
    
    # 生成 Markdown
    md_path = f"{OUTPUT_DIR}/{item['filename']}.md"
    generate_markdown(item['video'], transcript, 1, md_path)
    
    results.append({
        'video': item['video'],
        'markdown': md_path
    })
    
    print(f"  ✅ 完成: {md_path}")

print(f"\n📊 成功處理: {len(results)} 條影片")

## Step 6: 生成索引

生成 summary.md 索引檔案。

In [ ]:
# 生成索引
summary_path = f"{OUTPUT_DIR}/summary.md"

with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("# Bilibili 播放清單轉錄索引\n\n")
    f.write(f"**總影片數**: {len(results)}\n\n")
    f.write("## 影片列表\n\n")
    f.write("| 序號 | 標題 | UP主 |\n")
    f.write("|------|------|------|\n")
    
    for i, r in enumerate(results, 1):
        title = r['video']['title']
        uploader = r['video']['uploader']
        filename = os.path.basename(r['markdown'])
        f.write(f"| {i} | [{title}](./{filename}) | {uploader} |\n")

print(f"✅ 索引生成: {summary_path}")
print("\n📁 全部檔案已儲存喺 Google Drive:")
print(f"   {OUTPUT_DIR}")

## ✅ 完成！

所有 Markdown 檔案已生成並儲存喺你嘅 Google Drive：`/MyDrive/bilibili-projects/output/`